In [1]:
# import time
# import numpy as np

# try:
#     import cupy as cp
#     gpu_ok = True
# except Exception as e:
#     cp = None
#     gpu_ok = False
#     print("cupy import failed:", e)

# try:
#     import amd_cupy
#     print("amd_cupy imported OK")
# except Exception as e:
#     print("amd_cupy import failed:", e)

# def time_numpy_matmul(n, repeats=3):
#     a = np.random.random((n, n)).astype(np.float32)
#     b = np.random.random((n, n)).astype(np.float32)

#     _ = a @ b
#     t0 = time.perf_counter()
#     for _ in range(repeats):
#         c = a @ b
#     t1 = time.perf_counter()
#     return (t1 - t0) / repeats

# def time_gpu_matmul(n, repeats=3):
#     a = cp.random.random((n, n), dtype=cp.float32)
#     b = cp.random.random((n, n), dtype=cp.float32)

#     _ = a @ b
#     cp.cuda.Stream.null.synchronize()

#     t0 = time.perf_counter()
#     for _ in range(repeats):
#         c = a @ b
#     cp.cuda.Stream.null.synchronize()
#     t1 = time.perf_counter()
#     return (t1 - t0) / repeats

# def find_n(target=5.0):
#     n = 1024
#     while n <= 12288:
#         t = time_numpy_matmul(n, repeats=1)
#         print(f"NumPy trial: n={n}, time={t:.3f} s")
#         if t >= target:
#             return n
#         n = int(n * 1.3)
#         if n % 2:
#             n += 1
#     return n

# n = find_n(5.0)
# print(f"\nChosen n = {n}\n")

# np_t = time_numpy_matmul(n, repeats=3)
# print(f"NumPy average: {np_t:.3f} s")

# if gpu_ok:
#     gpu_t = time_gpu_matmul(n, repeats=3)
#     print(f"GPU average:   {gpu_t:.3f} s")
#     print(f"Speedup:       {np_t / gpu_t:.2f}x")
# else:
#     print("No working cupy module found.")

In [2]:
import time
import numpy as np

import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
os.environ["ROCR_VISIBLE_DEVICES"] = "0"

import cupy as cp

def time_numpy_matmul(n, repeats=3):
    a = np.random.random((n, n)).astype(np.float32)
    b = np.random.random((n, n)).astype(np.float32)
    _ = a @ b
    t0 = time.perf_counter()
    for _ in range(repeats):
        c = a @ b
    t1 = time.perf_counter()
    return (t1 - t0) / repeats

def time_gpu_matmul(n, repeats=3):
    a = cp.random.random((n, n), dtype=cp.float32)
    b = cp.random.random((n, n), dtype=cp.float32)
    _ = a @ b
    cp.cuda.Stream.null.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        c = a @ b
    cp.cuda.Stream.null.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / repeats

def find_n(target=5.0, max_n=9000):
    n = 1024
    last_ok = n
    while n <= max_n:
        t = time_numpy_matmul(n, repeats=1)
        print(f"NumPy trial: n={n}, time={t:.3f} s")
        if t >= target:
            return n
        last_ok = n
        n = int(n * 1.25)
        if n % 2:
            n += 1
    return last_ok

n = find_n()
print("Chosen n =", n)

np_t = time_numpy_matmul(n, repeats=3)
print(f"NumPy average: {np_t:.3f} s")

if cp is not None:
    gpu_t = time_gpu_matmul(n, repeats=3)
    print(f"GPU average:   {gpu_t:.3f} s")
    print(f"Speedup:       {np_t / gpu_t:.2f}x")

NumPy trial: n=1024, time=0.013 s
NumPy trial: n=1280, time=0.014 s
NumPy trial: n=1600, time=0.026 s
NumPy trial: n=2000, time=0.038 s
NumPy trial: n=2500, time=0.076 s
NumPy trial: n=3126, time=0.160 s
NumPy trial: n=3908, time=0.225 s
NumPy trial: n=4886, time=0.529 s
NumPy trial: n=6108, time=0.908 s
NumPy trial: n=7636, time=1.741 s
Chosen n = 7636
NumPy average: 1.540 s
GPU average:   0.128 s
Speedup:       12.07x
